<p><small><small>This Notebook is made available subject to the licence and terms set out in the <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README file</a>.</small></small></p>

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Prepare your Data for Building a Classification Model

<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_8/gdm_lab_8_2_classification_part_1.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

2-4 hours (depending on the amount of data cleaning necessary)

In this first lab, you will load, clean and format the data for training a text classification model. At this stage, you should have completed previous activities from the Capstone course. Specifically you should have:

1. Defined your classification task,
2. Downloaded or constructed your dataset,
3. Documented your dataset with a Data Card, and
4. Assessed the impact of your classifier.

In this and the next lab, you will build on the activities to turn your project plan into a working classification model.

Similarly, as in the previous activities, this lab also contains a worked example for every step. Consider the code for the worked example as a template for your own implementation. Note, however, that some steps may not be applicable to your project, and likewise, you may have to complete additional steps, so treat the template in this and the next lab just as a starting point and feel free to add or delete code cells to this notebook as you see fit.

## Overview

In this first part of implementing your capstone project, you will focus on preparing your data for training a text classification model.

### What you will learn

By the end of this first part of the capstone project, you will be able to:

* Clean and preprocess text data to prepare quality inputs for training a classification model.

* Split datasets into appropriate training and test sets for improved evaluation and generalization.

* Format data for fine-tuning an instruction-based LLM classifier.

### Tasks

In this lab, you will:

* Load the dataset that you prepared in previous activities.

* Clean individual datapoints, if necessary.

* Split the dataset into training and test sets.

* Format the data to match your classification needs and prepare it for LLM fine-tuning.

* Save the dataset in a format that is compatible with the machine learning pipeline.

The end result of this lab will be a processed dataset that you can use in the second part to train and evaluate your classification model.

### What you will use

This lab builds on materials from several previous courses. If you would like to refresh your knowledge on any of the following topics, you can find the relevant materials in the following courses:

- **General knowledge on LLMs and transformers** (Courses 01 Build Your Own Small Language Model, 03 Design and Train Neural Networks, and 04 Disccover The Transformer Architecture).

- **Data preparation** techniques (Course 02 Represent Your Text Data).

- **Formatting data for fine-tuning** (Course 05 Fine-Tune Your Model).

- **Splitting data** techniques (Course 03 Training a Neural Network).

## How to use Google Colaboratory (Colab)

Google Colaboratory (also known as Google Colab) is a platform that allows you to run Python code in your browser. The code is written in cells that are excuted on a remote server.

To run a cell, hover over a cell, and click on the `run` button to its left. The run button is the circle with the triangle (▶). Alternatively, you can also click on a cell and use the keyboard combination Ctrl+Return (or ⌘+Return if you are using a Mac).

To try this out, run the following cell. This should print today's day of the week below it.

In [ ]:
from datetime import datetime
print(f"Today is {datetime.today():%A}.")

Note that the order in which you run the cells matters. When you are working through a lab, make sure to always run all cells in order. Otherwise, the code might not work. If you take a break while working on a lab, Colab may disconnect you and in that case, you have to execute all cells again before  continuing your work. To make this easier, you can select the cell you are currently working on and then choose __Runtime → Run before__  from the menu above (or use the keyboard combination Ctrl/⌘ + F8). This will re-execute all cells before the current one.

### Using Colab with a GPU

Follow these steps to run the activities in this lab on a GPU:

1.  In the top menu bar, click on **Runtime**.
2.  Select **Change runtime type** from the dropdown menu.
3.  In the pop-up window under **Hardware Accelerator**, select **GPU** (usually listed as `T4 GPU`).
5.  Click **Save**.

Your Colab session will now restart with GPU access.

Note that access to GPUs is limited and at times, you may not be able to run this lab on a GPU. All activities will still work but they will run slower and you will have to wait longer for some of the cells to finish running.


## Imports

The following cell contains imports for a range of libraries that you may use for preparing your dataset. Depending on your implementation, you may need to import additional libraries.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill # Text wrapping utilities.

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

from sklearn.model_selection import train_test_split # Data splitting utils.
import matplotlib.pyplot as plt # Plotting and visualisation.

# Deep learning and AI libraries.
import jax.numpy as jnp # JAX numerical computing.
import keras # High-level neural networks API.
import keras_nlp # Keras NLP extensions.

# Google Colab specific.
from google.colab import userdata # Access to Colab secrets.
from google.colab import drive # Access to Google Drive.

## Load your data

As a first step, you will have to load the dataset that you created in the previous activity, so that you can preprocess your data. One method to do this is to upload the dataset to Google Drive and then connect your Google Drive to Colab.

### Connect Colab with Google Drive

The following cell mounts your Google Drive so that you can access files by adding `/content/drive/MyDrive` to the beginning of your file path. Running this cell will open a dialog that asks you whether you want to connect Colab to Google Drive. Follow the instructions and enable all permissions, so that Colab can read and write files from Google Drive.

In [ ]:
drive.mount('/content/drive')

### Activity 1: Load, inspect, and filter your data



------
> 💻 **Your task:**
>
> Load and inspect your dataset by adding your code to the subsequent  cells:
>
> 1. **Define the path to your dataset**: Set the path to your own dataset file path.
>
> 2. **Choose a data loading method** to match your data format:
>    - For CSV files: `pd.read_csv(data, encoding="utf-8")`.
>    - For JSON files: Use `json.load()` or `pd.read_json(data)`.
>    - For JSONL files: Use `pd.read_json(data, lines=True)`.
>    - For API data: Implement your own loading function.
>
> 3. **Verify your data structure**: After loading, check that:
>    - Your text data is in an appropriate column.
>    - Your labels/categories are correctly formatted.
>    - The encoding is preserved (especially for non-English text).
>
> 4. **Inspect the data**: Use `df.head()`, `df.info()`, and `df.columns` to understand your dataset structure and size.
>    - Identify your classification property.
>
> 5. **Define your classes**:
>    - Keep classes as **mutually exclusive** as possible.
>    - Try to make the classes **well-defined** and distinguishable.
>    - Aim for **balanced representation** in your training data.
>
> 6. **Filter and balance your data**:
>    - Define a `selected_categories` list to include only the categories relevant to your task (if applicable).
>    - Adjust the cap per category (default: 100 items) based on your dataset size and available compute resources (if applicable).
>    - Larger datasets require more training time but may improve model performance.
>
> The goal of these steps is to bring all records into a variable (a Pandas `DataFrame`) so that you can explore the structure, check the columns, preview a few rows, and confirm the encoding, before moving on to transformation for LoRA training.
>
------


#### Worked example: Load and inspect your data



```python
# Open the file from Google Drive.
filepath = "/content/drive/MyDrive/africa_galore_large_classification.jsonl"

# Read with pandas.
df = pd.read_json(filepath, lines=True)
# Other file-type examples:
# df = pd.read_json(filepath)
# data = pd.read_csv(filepath, encoding="utf-8")
# data = json.load(open(filepath))
# data = load_from_api()

# Take a quick look at the first few rows.
display(df.head())

print("Columns:", df.columns.tolist())
print("Number of rows:", len(df))

# Display unique values from the 'category' column.
print("Unique categories:", df["category"].unique())
```

#### Your implementation

In [ ]:
# Add your code for defining the path to your dataset here.

# Add your code for loading your dataset here.

# Add your code for printing the first few examples in your dataset here.


#### Filter and balance the dataset

Depending on the dataset you are using you may have too many examples (which could make training your classifier very slow) or your data may contain more categories than you need, or the dataset may be unbalanced. For example, in the worked example using the Africa Galore dataset, the dataset contains thousands of records across many categories. However, for the purpose of this worked example, only a subset of well-defined categories is considered and retained for training the classification model: **Drink**, **Food**, **Music** and **Textile**. The number of examples in each category is also capped to **100 items** to create a balanced training set.

------
> **ℹ️ Info: Why balance the dataset?**
>
> When training a classifier, having roughly equal numbers of examples per category helps prevent **class imbalance bias**. If one category has significantly more examples than others, the model may learn to favour the majority class and underperform on minority classes. Capping each category to 100 items helps the model receive balanced exposure to all categories during fine-tuning, leading to more reliable classification across all classes.
------

#### Worked example: Filter and balance the dataset

```python
# Define the categories to keep.
selected_categories = [
    "Drink",
    "Food",
    "Music",
    "Textile",
]

# Filter the dataframe to include only selected categories.
df = df[df["category"].isin(selected_categories)].copy()

# Cap to 100 rows per category.
df = (
    df.sort_index() # Group by category at this stage.
      .groupby("category", group_keys=False)
      .head(100)
)

# Shuffle the data to reinforce random ordering during training later.
# This prevents the model from learning spurious patterns based on data order.
# random_state ensures reproducibility.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Validate the filtered data.
print("Filtered dataset:")
display(df.head())

print("\nColumns:", df.columns.tolist())
print("Number of rows:", len(df))
print("Unique categories:", df["category"].unique())
```

#### Your implementation

In [ ]:
# Add you code to define categories to keep (if applicable).

# Add your code to filter your dataset (if applicable).

# Add your code to balance your dataset (if applicable).


## Activity 2: Clean the data


------
> 💻 **Your task:**
>
> Clean and prepare your text data for fine-tuning:
>
> 1. **Assess your data quality**: Examine your text data for:
>    - HTML tags, markup, or special formatting.
>    - Non-text characters (emojis, symbols, special Unicode).
>    - Inconsistent encoding or corrupted characters.
>    - Excessive whitespace or formatting issues.
>
> 2. **Apply appropriate cleaning functions**:
>    - Use `clean_html()` to remove HTML tags and entities.
>    - Use `clean_unicode()` to remove emojis and non-text symbols.
>    - Add custom cleaning functions if needed for your domain.
>
> 3. **Preserve original data**: Keep a copy of your original text before cleaning, for example, if your input text is stored in the field `description`, then you can make a copy as follows:
>    ```python
>    df['description_original'] = df['description'].astype(str)
>    ```
>
> 4. **Preview cleaning results**: Compare before/after examples to ensure:
>    - Important content is preserved.
>    - Unwanted elements are removed.
>    - Text remains readable and coherent.
>
> 5. **Decide on final text column**: Choose whether to:
>    - Replace original text with cleaned version.
>    - Keep both versions for comparison.
>    - Apply different levels of cleaning for different purposes.
>
> **Quality check**: Examine several examples to verify that cleaning improves data quality without losing essential information.
>
------

Refer to Course 02 Represent Your Text Data for examples on how to adjust and clean your data. The following code provides utility functions from Course 02 Represent Your Text Data to clean texts. You may want to add your own cleaning functions or modify the ones below depending on what kind of cleaning your data requires.

In [ ]:
def clean_html(text: str) -> str:
    """
    Strip basic HTML markup and common entities from a string.

    The funcion does **not** attempt full HTML parsing; for more complex markup
    consider ``BeautifulSoup`` or ``html.unescape``.

    Args:
      text: The text string that may contain HTML tags or entities.

    Returns:
      A cleaned string with tags stripped and the entities '&nbsp;', '&amp;',
        '&lt;' and '&gt;' converted to ' ', '&', '<' and '>'.
    """

    # Remove HTML tags.
    text = re.sub(r"<.*?>", "", text)

    # Replace HTML entities.
    text = re.sub("&nbsp;", " ", text)  # Replace non-breaking space.
    text = re.sub("&amp;", "&", text)  # Replace "&amp;" with "&".
    text = re.sub("&lt;", "<", text)  # Replace "&lt;" with "<".
    text = re.sub("&gt;", ">", text)  # Replace "&gt;" with ">".

    return text


def clean_unicode(text: str) -> str:
    """
    Removes non-text unicode characters from a string.

    Args:
      text: The original text which may contain special characters.

    Returns:
      The input text string with emojis and other non-text symbols removed.
    """

    categories_to_keep = {"L", "N", "P"}  # L=letters, N=numbers, P=punctuation.

    keep = []
    for ch in text:
        do_keep = ch.isspace()  # Preserve spaces.
        if not do_keep:
            for category in categories_to_keep:
                if unicodedata.category(ch).startswith(category):
                    do_keep = True
                    break
        if do_keep:
            keep.append(ch)
    return "".join(keep)

#### Worked example: Cleaning data in a DataFrame

```python
# Clean the target text column and preview before/after.
# In this example, 'description' is the primary input for classification.
# If your names contain HTML or special characters, apply cleaning here.

# Keep an original copy for reference.
df["description_original"] = df["description"].astype(str)
df["description_clean"] = (
    df["description_original"].map(clean_html).map(clean_unicode)
)

# Quick preview.
preview_cols = [
    c
    for c in [
        "category",
        "description_original",
        "description_clean",
    ]
    if c in df.columns
]
display(df[preview_cols].head(5))

# Replace the original name with the cleaned version here.
df["description"] = df["description_clean"]
```

#### Your implementation

In [ ]:
# Add your code for cleaning texts in your dataset here.


## Activity 3: Split the original data


------
> 💻 **Your task:**
>
> Create a validation and a test set for evaluating your model during model development and for performing a final evaluation.
>
> If you have not included all examples from your dataset, you can choose examples from your dataset that you have not included in the training data to be used for the validation and test sets (as done in the worked example).
>
> Alternatively, if you included all examples from your dataset in the training set, you can use stratified splitting to maintain class proportions:
>```python
>      df_train, df_val_test = train_test_split(
>          df, test_size=0.2, random_state=42, stratify=df['category']
>      )
>      # Split into validation and test sets.
>      df_val, df_test = train_test_split(
>          df_val_test, test_size=0.5, random_state=42, stratify=df['category']
>      )
> ```
> <br>
>
> **Important considerations**:
>   - **Performance warning**: Gemma3-1B may perform poorly on completely unknown test examples if your training dataset is small.
>   - For small datasets, consider using more training data and smaller test sets.
>   - Ensure evaluation data reflects your actual use case and domain.
>
------

-----
> **ℹ️ Info: Training, validation, and test sets**
>
> In machine learning, data is typically divided into three parts: a training set, a validation set, and a test set.
>
> The **training set** is used to train the model by adjusting its internal parameters. The **validation set** is used during development to fine-tune hyperparameters, compare model versions, and prevent overfitting by monitoring how well the model generalizes to unseen data. The **test set** is held out until the very end and provides an unbiased estimate of the model's final performance on entirely new data.
>
>
> Revisit Course 03 Training a Neural Network to review the article on  splitting data. For more information also see [Google's *Dividing the original dataset* crash course](https://developers.google.com/machine-learning/crash-course/overfitting/dividing-datasets/).
>
-----

-----
> **ℹ️ Info: Stratified splitting**
>
> Stratified splitting ensures that the proportion of samples from each class (category) is preserved in both the training and test sets.
>
> For example, if your original dataset has 60% 'Food' items and 40% 'Music' items, stratified splitting will maintain these same proportions in both the training and test sets.
>
> This helps achieve reliable evaluation results, especially when you have imbalanced classes.
>
> For more information, see the 🔗[scikit-learn documentation on stratification](https://scikit-learn.org/stable/modules/cross_validation.html#stratification).
>
-----

#### Worked example: Create a test set from unused examples

Since the training data was created by taking the first 100 items per category (sorted by index) from the original dataset, you can create a test set by loading the original data and selecting items that were **not** included in the training set.

The code below:
1. Reloads the original `africa_galore_large.json` dataset.
2. Filters to the same categories used for training.
3. Excludes the first 100 items per category (which were used for training).
4. Samples balanced validation and test sets from the remaining unseen items.

This compiles validation and test split with items the model has **never seen** during fine-tuning, providing a more accurate evaluation of generalization performance.

```python
# Load the original full dataset.
url = "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_large.json"
df_full = pd.read_json(url, lines=True)

# Filter to selected categories.
# Use the same categories as the training data (selected_categories).
df_filtered = df_full[
    df_full["category"].isin(selected_categories)
].copy()

# For each category, exclude the first 100 items (used for training)
# and keep the rest as potential test data.
df_unseen = (
    df_filtered
    .sort_index()
    .assign(row_in_cat=lambda x: x.groupby("category").cumcount())
    .query("row_in_cat >= 100")
    .drop(columns="row_in_cat")
)

# Sample a balanced test set (here, 40 items per category).
test_samples_per_category = 40

df_val_test = (
    df_unseen
    .groupby("category", group_keys=False)
    .sample(
        n=test_samples_per_category,
        random_state=42,
        replace=False
    )
    .reset_index(drop=True)
)

# Repeat the cleaning processs from above.
df_val_test["description_original"] = df_val_test["description"].astype(str)
df_val_test["description_clean"] = (
    df_val_test["description_original"].map(clean_html).map(clean_unicode)
)

# Replace the original name with the cleaned version here.
df_val_test["description"] = df_val_test["description_clean"]

# Split into validation and test sets.
df_val, df_test = train_test_split(
    df_val_test,
    test_size=0.5,
    random_state=42,
    stratify=df_val_test["category"]
)

# Validate the validation set.
print(f"Validation set created with {len(df_val)} unseen items.")
print("\nItems per category:")
print(df_val["category"].value_counts())
display(df_val.head())

# Validate the test set.
print(f"Test set created with {len(df_test)} unseen items.")
print("\nItems per category:")
print(df_test["category"].value_counts())
display(df_test.head())
```

#### Your implementation:

In [ ]:
# Add your code for creating a validation and a test split here.


## Activity 4: Format the data


------
> 💻 **Your task:**
>
> Convert your dataset into a format that can be used to fine-tune a Gemma model.
>
> 1. **Define a `to_instruction_record` function**. This function should turn an example from your dataset into a pair of input and output that can be used to fine-tune a Gemma model.
>
> 2. **Design your instruction prompt**: Create a clear, specific instruction that tells the model what to do:
>    - Example: "Classify the following review as positive, negative, or neutral:".
>    - Example: "Determine the intent of this customer message:".
>    - Include your class list in the prompt if helpful for clarity.
>
> 3. **Test the transformation**: Run the code to transform your data and inspect the first record to ensure it looks correct.
>
> 4. **Save your training data**: Save your data to Google Drive for fine-tuning.
>
> **Verification**: Check that your JSONL file has the correct format with `input` and `output` fields for each record.
>
> Take a look at the worked example, which implements all of these steps and adapt it to your dataset.
------



-------
> **ℹ️ Info: Data format for fine-tuning Gemma**
>
>Fine-tuning Gemma typically requires pairs of input and output, where each line of the training file is a standalone JSON object containing:
>
>* **`input`** – the text that the model will read and classify.  
> *The `name` columns from the Africa Galore dataset are used in the sample code as input data.*
>* **`output`** – the correct label the model should generate.  
>  *The `category` column (e.g., 'Food', 'Music') serves as the output label in this example.*
>
>During training, the model learns to map from the `input` field to the correct `output` label.
--------

#### Worked example: Format the data

```python
def to_instruction_record(row) -> dict[str, str]:
   """Transform a single dataset row into instruction-tuning format.

    Args:
      row: A single row from a Pandas DataFrame.

    Returns:
      dict: A dictionary with "input" and "output" as keys.
    """

    instruction = (
        "Classify the following text into one of the categories 'Drink',"
        " 'Food', 'Music', or 'Textile'.\n\n"
    )

    return {
        "input": (
            instruction + row["description"]
        ),
        "output": row["category"],
    }


# Apply the transformation.
instruction_records_train = [to_instruction_record(r) for _, r in df.iterrows()]
instruction_records_val = [
    to_instruction_record(r) for _, r in df_val.iterrows()
]
instruction_records_test = [
  to_instruction_record(r) for _, r in df_test.iterrows()
]

# Preview a single transformed record.
print(json.dumps(instruction_records_train[0], ensure_ascii=False, indent=2))

# Save to a JSONL file for fine-tuning.
with open(
    "/content/drive/MyDrive/classification_training_data.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for rec in instruction_records_train:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(
    "/content/drive/MyDrive/classification_validation_data.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for rec in instruction_records_val:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(
    "/content/drive/MyDrive/classification_test_data.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for rec in instruction_records_test:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
 ```

#### Your implementation

In [ ]:
def to_instruction_record(row) -> dict[str, str]:
    """Transform a single dataset row into instruction-tuning format.

    Args:
      row: A single row from a Pandas DataFrame.

    Returns:
      dict: A dictionary with "input" and "output" as keys.
    """

    instruction = "" # Add your instruction here.

    return {
        "input": instruction + "",  # Add your input here.
        "output": "",  # Add your output here.
    }


# Add your code for applying the transformation.


# Add your code for printing a single transformed record.


# Add your code for saving to a JSONL file for fine-tuning.



## Summary

In the first part of this capstone pathway, you prepared the dataset so that you can use it for fine-tuning an LLM for a classification task. This involved loading the dataset, cleaning the data, splitting it into a training, development, and test split, and formatting it to be used for fine-tuning.

In the second part of this captstone pathway, you will use these preprocessed data splits to fine-tune a model for classification.